# 08. Group-aware sampling and semantic phase origins

![Source groups assigned to disjoint partitions before they become windows, transforms, and a lineage record](../images/08_group_aware_sampling.svg)

![A source sequence supplying overlapping windows, with one crop and flip drawn per window rather than per frame](../images/08_windows_and_transforms.svg)

Lesson 07 fixed how much optimization every model in the study receives. This notebook builds a small synthetic version of the thing that optimization consumes: the phase catalog of the iso-catalog study.

The order of the cells is the order of the decisions. Split groups, place nested phase origins inside the surviving sequences, build the matched nearby-jitter control, check that the origins really are more separated than the jitter, confirm the allocation arithmetic, and store the audit alongside the samples.

Read the [lecture](../lectures/08_group_aware_sampling.md) for the reasoning behind each step, and return to the [tutorial index](../README.md) when you are done.

**Learning goals:** split groups before sampling any clip, select nested phase origins from a common eligible corpus, construct a matched nearby-jitter control, and audit semantic separation instead of counting start indices.

One idea underlies all four: a count of windows is not a measure of evidence. Everything below either protects that distinction or measures it.

In [ ]:
import hashlib
import numpy as np

SEED = 23
rng = np.random.default_rng(SEED)
assert rng.random() >= 0.0


## Groups first

Every window from one source sequence has to land in one partition. If it does not, the test set contains the same background, body, and camera as the training set, and the score stops being about movement.

The cell below makes 12 synthetic groups with 5 rows each, holds out three whole groups, and asserts that the two sides share no group at all. These identifiers stand in for source groups, not for people: calling them people would be a claim about the data that nothing here has verified.

The `assert` is the real content of the cell. A disjointness check is cheap, it runs in a second, and it catches the single most expensive mistake in this entire pipeline.

In [ ]:
groups = np.repeat(np.arange(12), 5)
test_groups = {1, 4, 8}
train = groups[~np.isin(groups, list(test_groups))]
test = groups[np.isin(groups, list(test_groups))]
assert set(train).isdisjoint(set(test))


## Nested semantic origins

With the split settled, the question moves inside a sequence: where in the walk should a clip start? Phase is the coordinate that answers it. Phase 0 and phase 1 are the same point of the gait cycle, so an origin is a fraction of one stride.

`stable_uniform` hashes a namespace, the sequence ID, and the replicate block into a number in the unit interval. That base phase depends only on identity, never on a row's position in a manifest, so a smaller pool that is a subset of a larger one keeps exactly the same base phases.

The origin sets are nested by construction: the offsets for `k = 1`, `2`, and `4` are prefixes of one another, so adding origins adds views without moving the first one. `nearby_jitter` keeps the same base phase and the same count of four, and only pulls the four origins into a tight cluster. That is what makes it a control rather than a fourth condition: it changes separation and nothing else.

The two asserts state both properties. The first origin is shared across `k`, and the jitter set has the same size as the semantic set.

In [ ]:
def stable_uniform(*parts):
    payload = '|'.join(map(str, parts)).encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], 'big') / 2**64

def semantic_origins(sequence_id, block, k):
    base = stable_uniform('phase-v1', sequence_id, block)
    offsets = {1: (0.0,), 2: (0.0, 0.5), 4: (0.0, 0.25, 0.5, 0.75)}[k]
    return tuple((base + offset) % 1.0 for offset in offsets)

def nearby_jitter(sequence_id, block, radius=0.04):
    base = stable_uniform('phase-v1', sequence_id, block)
    return tuple((base + offset) % 1.0 for offset in (-radius, -radius / 3, radius / 3, radius))

one = semantic_origins('s07', 0, 1)
two = semantic_origins('s07', 0, 2)
four = semantic_origins('s07', 0, 4)
jitter = nearby_jitter('s07', 0)
assert one[0] == two[0] == four[0]
assert len(four) == len(jitter) == 4


## Audit the intended separation

Defining separated origins is not the same as demonstrating them, so the next cell measures what the previous one built.

`circular_distance` measures the gap between two phases the short way around the cycle, so 0.9 and 0.05 are 0.15 apart rather than 0.85. `trajectory_separation` averages that distance over every unordered pair of origins in a set.

The quarter-cycle origins average one third of a cycle apart, while jitter with a radius of 0.04 averages about 0.044, roughly seven times smaller. The `assert` demands that ordering, because if the two conditions were not measurably different there would be nothing to compare.

Read this as a synthetic proxy and nothing more. The real audit uses frozen silhouette features and blinded manual validation, and if that audit fails the phase branch stops.

In [ ]:
def circular_distance(a, b):
    delta = abs(a - b) % 1.0
    return min(delta, 1.0 - delta)

def trajectory_separation(origins):
    pairs = [circular_distance(a, b) for i, a in enumerate(origins) for b in origins[i + 1:]]
    return float(np.mean(pairs))

semantic_sep = trajectory_separation(four)
jitter_sep = trajectory_separation(jitter)
assert semantic_sep > jitter_sep
print('semantic separation', semantic_sep, 'jitter separation', jitter_sep)


## Iso-catalog allocation

Now assemble whole conditions from the pieces above. Each cell of the study pairs a number of unique sequences with a number of origins per sequence, and every product is the same: 250,000 sequence-origin pairs.

The three path cells move diversity from across sequences to within sequences at constant cardinality. `nearby_jitter` matches `phase_depth` on both numbers and differs only in the origin policy, which is exactly the pairing the audit above supports.

Equal cardinality controls counting, not information content. Four origins in one sequence are not four new sequences, and the loop's `assert` proves arithmetic, not equivalence.

The last two lines hash the frozen catalog settings into a digest. That digest is what later stages compare, so a changed threshold or policy becomes a visible mismatch instead of an invisible drift.

In [ ]:
allocations = {
    'breadth': (250_000, 1, 'base_phase'),
    'balanced': (125_000, 2, 'phase_separated'),
    'phase_depth': (62_500, 4, 'phase_separated'),
    'nearby_jitter': (62_500, 4, 'nearby_jitter'),
}
for name, (unique_sequences, origins_per_sequence, policy) in allocations.items():
    nominal_catalog_size = unique_sequences * origins_per_sequence
    assert nominal_catalog_size == 250_000
    print(name, policy, nominal_catalog_size)

phase_catalog = {'version': 'phase-v1', 'policy': 'phase_separated', 'threshold': 0.12}
phase_catalog_digest = hashlib.sha256(repr(sorted(phase_catalog.items())).encode()).hexdigest()
assert len(phase_catalog_digest) == 64


## Preserve the audit with the sample

A treatment label such as `phase_separated` records an intention. It does not record what actually happened, and months later the intention is all anyone can see.

So store the measurements next to the label. `required_audit_fields` fixes the schema: the catalog digest, the origin policy, the nominal catalog size, origin coverage, window overlap, the measured trajectory separation, and the effective near-duplicate cluster count. Comparing the record's keys with `==` rejects both a missing field and an unplanned extra one.

The final assertion repeats the separation check against the stored value rather than a local variable. That stored audit record is what travels with the data, and an audit is only useful if the number that travels with the data is the one that was checked.

In [ ]:
required_audit_fields = {
    'phase_catalog_digest', 'origin_policy', 'nominal_catalog_size',
    'origin_coverage', 'window_overlap', 'trajectory_separation',
    'effective_cluster_count',
}
audit_record = {
    'phase_catalog_digest': phase_catalog_digest, 'origin_policy': 'phase_separated',
    'nominal_catalog_size': 250_000, 'origin_coverage': 4,
    'window_overlap': 0.5, 'trajectory_separation': semantic_sep,
    'effective_cluster_count': 61_000,
}
assert set(audit_record) == required_audit_fields
assert audit_record['trajectory_separation'] > jitter_sep


**Takeaway:** group-aware splitting prevents leakage, and a phase intervention needs more than that. It needs one common eligibility rule, nested origin sets built from stable identity, a matched jitter control, and an audit that demonstrates the separation instead of asserting it.

Lesson 09 takes the representations this pipeline produces and asks what shape they have.

Previous: [07. Gradient updates](07_gradient_updates_and_schedules.ipynb) · Next: [09. Eigenspectra](09_eigenspectra_and_effective_rank.ipynb)